Inspired by Bereta & Diamantis (2025), "The Shape of Consumer Behavior:
A Symbolic and Topological Analysis of Time Series"

This notebook is the clustering-focused version of `consumer_behavior_pipeline.py`, organized to match the format/configuration style of `event_eda.ipynb`.

Pipeline stages
----------------
1. Data loading - reads fixed stitched Google Trends CSVs from each term folder
2. KS test - Kolmogorov-Smirnov test for normality
3. Descriptive statistics - distributions, volatility, top average interest
4. STL + ADF - decomposition and stationarity checks
5. SAX / eSAX - symbolic time-series representations
6. TDA - persistence landscapes from sliding-window embeddings
7. Clustering - KMeans + Ward hierarchical clustering
8. Output - CSVs and plots for cluster review


In [13]:
from __future__ import annotations

import warnings
from pathlib import Path
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

from scipy import stats
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.preprocessing import LabelEncoder
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller

warnings.filterwarnings("ignore")

# ----------------------------------------------------------------------------
# CONFIG -- edit these for your environment
# ----------------------------------------------------------------------------

DATA_DIR = Path(r"C:\Python\CSUREMM\data_events")
OUTPUT_DIR = Path(r"C:\Python\CSUREMM\output")

# Stitched files live at: DATA_DIR/<term>/stitched/gt_fixed_<term>_<start>_<end>.csv
# (NOT directly inside the term folder, and the filename embeds the term name
# itself plus a date range that varies, so it can't be matched by a fixed
# literal filename -- it's located via the "stitched" subfolder + glob below.)
STITCHED_SUBDIR = "stitched"
STITCHED_GLOB = "gt_fixed_*.csv"

# --- SAX / eSAX parameters (rescaled for daily data; see module docstring) ---
WINDOW_SIZE = 365     # sliding window length, in data points (paper: 52 weekly)
N_SEGMENTS = 12        # PAA segments per window (paper: 12)
ALPHABET_SIZE = 4      # number of SAX symbols (paper uses 4: a,b,c,d)
SAX_STEP = 1            # slide step in points (paper: 1)

# --- TDA parameters (rescaled for daily data; see module docstring) ---
EMBED_DIM = 6           # embedding dimension d (paper: 6)
EMBED_TAU = 21          # time delay tau, in data points (paper: 3 weekly -> 21 daily)
MAX_HOMOLOGY_DIM = 1    # compute H0 and H1
N_LANDSCAPE_POINTS = 200  # discretization resolution for persistence landscapes
N_LANDSCAPE_LAYERS = 5    # number of landscape layers (k=1..5) to keep

# --- Clustering ---
N_CLUSTERS = 6          # paper used k=6 throughout (elbow method)
RANDOM_STATE = 42

# --- STL decomposition / ADF test (paper Section 3.2-3.4) ---
ROLLING_WINDOW = 91          # 13-week rolling avg (paper) -> 13*7 = 91 days
STL_PERIOD = 365              # seasonal period: paper used weekly seasonality
                               # implicitly tied to ~annual cycle; daily data ->
                               # 365-day period captures the same annual seasonality
ADF_ALPHA = 0.05               # significance level for ADF stationarity test

ALPHABET = list("abcdefghijklmnopqrstuvwxyz"[:ALPHABET_SIZE])


In [14]:
# ----------------------------------------------------------------------------
# 1. DATA LOADING
# ----------------------------------------------------------------------------

def load_all_series(
    data_dir: Path,
    stitched_subdir: str = STITCHED_SUBDIR,
    filename_glob: str = STITCHED_GLOB,
) -> dict[str, pd.Series]:
    """
    Walk every term subfolder in data_dir, read its stitched daily csv from
    DATA_DIR/<term>/<stitched_subdir>/<filename_glob>, and return
    {term_name: pandas Series indexed by date}.
    """
    series_dict: dict[str, pd.Series] = {}
    failed: list[tuple[str, str]] = []

    for folder in sorted(data_dir.iterdir()):
        if not folder.is_dir():
            continue

        stitched_dir = folder / stitched_subdir
        candidates = sorted(stitched_dir.glob(filename_glob))

        # Fallbacks make the notebook tolerant to older output names.
        if not candidates:
            candidates = sorted(stitched_dir.glob("gt_stitched_fixed_*.csv"))
        if not candidates:
            candidates = sorted(stitched_dir.glob("gt_stitched_*.csv"))

        if not candidates:
            failed.append((folder.name, f"missing file matching {filename_glob} in {stitched_dir}"))
            continue

        fpath = candidates[-1]

        try:
            df = pd.read_csv(fpath, parse_dates=["Time"])
            value_col = [c for c in df.columns if c != "Time"][0]
            ts = (
                df.set_index("Time")[value_col]
                .sort_index()
                .astype(float)
            )
            ts.name = folder.name
            series_dict[folder.name] = ts
        except Exception as e:
            failed.append((folder.name, str(e)))

    if failed:
        print(f"[load_all_series] {len(failed)} terms failed to load:")
        for name, err in failed:
            print(f"    {name}: {err}")
    print(f"[load_all_series] Loaded {len(series_dict)} term series.")
    return series_dict


def build_panel(series_dict: dict[str, pd.Series]) -> pd.DataFrame:
    """Align all series on their shared date index (outer join), as in paper Fig.5/Table 1."""
    panel = pd.DataFrame(series_dict)
    return panel


def summary_statistics(panel: pd.DataFrame) -> pd.DataFrame:
    """Replicates paper Table 1 (count, mean, std, min, 25/50/75pct, max) for every term."""
    desc = panel.describe(percentiles=[0.25, 0.5, 0.75]).T
    desc = desc.rename(columns={
        "count": "Count", "mean": "Mean", "std": "Std", "min": "Min",
        "25%": "25%", "50%": "50%", "75%": "75%", "max": "Max",
    })
    return desc[["Count", "Mean", "Std", "Min", "25%", "50%", "75%", "Max"]]


In [15]:
# ----------------------------------------------------------------------------
# 2. KOLMOGOROV-SMIRNOV TEST FOR NORMALITY (paper Table 3 / Section 3)
# ----------------------------------------------------------------------------

def ks_normality_test(panel: pd.DataFrame, alpha: float = 0.05) -> pd.DataFrame:
    """
    For each term's full raw series, run a one-sample KS test against a
    normal distribution fitted to that series' own mean/std (matching the
    paper's approach in Section 3 / Table 3).
    """
    rows = []
    for term in panel.columns:
        x = panel[term].dropna().values
        mu, sigma = x.mean(), x.std(ddof=1)
        if sigma == 0:
            stat, p = np.nan, np.nan
        else:
            stat, p = stats.kstest(x, "norm", args=(mu, sigma))
        rows.append({
            "Keyword": term,
            "KS Statistic": stat,
            "p-value": p,
            "Reject H0 at 5%": "Yes" if (p is not np.nan and p < alpha) else "No",
        })
    out = pd.DataFrame(rows).sort_values("KS Statistic", ascending=False).reset_index(drop=True)
    return out


In [16]:
# ----------------------------------------------------------------------------
# 2b. DESCRIPTIVE STATISTICS  (paper Section 3.3: boxplots, volatility, top interest)
# ----------------------------------------------------------------------------

def descriptive_statistics_extras(panel: pd.DataFrame) -> dict[str, pd.DataFrame]:
    """
    Replicates paper Section 3.3 / Figure 9 / Figure 10:
      - per-keyword distribution (boxplot data already in summary_statistics)
      - top-5 most volatile keywords (by std)
      - top-5 keywords by highest average interest (by mean)
    """
    stds = panel.std().sort_values(ascending=False)
    means = panel.mean().sort_values(ascending=False)

    top5_volatile = stds.head(5).rename("Std").to_frame()
    top5_interest = means.head(5).rename("Mean").to_frame()

    return {
        "stds_all": stds.rename("Std").to_frame(),
        "means_all": means.rename("Mean").to_frame(),
        "top5_volatile": top5_volatile,
        "top5_interest": top5_interest,
    }


def plot_boxplots(panel: pd.DataFrame, outpath: Path, terms_per_row: int = 30):
    """
    Replicates paper Figure 9. With 150 terms, a single boxplot row would be
    unreadable, so terms are split into chunks of `terms_per_row` and plotted
    across multiple stacked subplots instead of one wide figure.
    """
    terms = list(panel.columns)
    n_chunks = int(np.ceil(len(terms) / terms_per_row))
    fig, axes = plt.subplots(n_chunks, 1, figsize=(16, 4 * n_chunks))
    if n_chunks == 1:
        axes = [axes]
    for i in range(n_chunks):
        chunk = terms[i * terms_per_row:(i + 1) * terms_per_row]
        ax = axes[i]
        ax.boxplot([panel[t].dropna().values for t in chunk], label=chunk, showfliers=True)
        ax.set_xticklabels(chunk, rotation=90, fontsize=7)
        ax.set_ylabel("Interest (0-100)")
    fig.suptitle("Boxplot of Interest per Keyword")
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()


def plot_top5_volatility_interest(stats_extras: dict[str, pd.DataFrame], outpath: Path):
    """Replicates paper Figure 10: top-5 volatile keywords + top-5 highest-interest keywords."""
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    vol = stats_extras["top5_volatile"].iloc[::-1]
    axes[0].barh(vol.index, vol["Std"], color="tab:orange")
    axes[0].set_xlabel("Standard Deviation")
    axes[0].set_title("Most Volatile Keywords")

    interest = stats_extras["top5_interest"].iloc[::-1]
    axes[1].barh(interest.index, interest["Mean"], color="tab:blue")
    axes[1].set_xlabel("Mean Interest")
    axes[1].set_title("Highest Average Interest")

    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()


In [17]:
# ----------------------------------------------------------------------------
# 2c. SEASONAL-TREND DECOMPOSITION (STL)  (paper Section 3.4)
# ----------------------------------------------------------------------------

def run_stl_decomposition(series: pd.Series, period: int = STL_PERIOD) -> STL:
    """
    Runs STL (Seasonal-Trend decomposition using LOESS) on a single series.
    Returns the fitted DecomposeResult (has .trend, .seasonal, .resid, .observed).
    Requires at least 2 full periods of data; raises if the series is too short.
    """
    x = series.dropna()
    if len(x) < 2 * period:
        raise ValueError(
            f"Series too short for STL with period={period} "
            f"(need >= {2 * period} points, have {len(x)})."
        )
    stl = STL(x, period=period, robust=True)
    return stl.fit()


def run_stl_for_all(series_dict: dict[str, pd.Series], period: int = STL_PERIOD,
                     outdir: Path | None = None) -> dict[str, object]:
    """
    Runs STL decomposition for every term. Optionally saves a 4-panel plot
    (observed/trend/seasonal/resid) per term, mirroring paper Figure 11.
    Returns dict[term] -> statsmodels DecomposeResult.
    """
    results = {}
    failed = []
    return results

def stl_summary_table(stl_results: dict[str, object]) -> pd.DataFrame:
    """
    Summarizes STL output per term: variance explained by trend/season/resid,
    useful as a compact numeric companion to the per-term plots.
    """
    rows = []
    for term, res in stl_results.items():
        observed_var = np.nanvar(res.observed)
        trend_var = np.nanvar(res.trend)
        season_var = np.nanvar(res.seasonal)
        resid_var = np.nanvar(res.resid)
        rows.append({
            "Keyword": term,
            "Observed Var": observed_var,
            "Trend Var": trend_var,
            "Seasonal Var": season_var,
            "Residual Var": resid_var,
            "Seasonal Strength": max(0, 1 - resid_var / (season_var + resid_var))
                                 if (season_var + resid_var) > 0 else np.nan,
            "Trend Strength": max(0, 1 - resid_var / (trend_var + resid_var))
                               if (trend_var + resid_var) > 0 else np.nan,
        })
    return pd.DataFrame(rows).sort_values("Keyword").reset_index(drop=True)


In [18]:
# ----------------------------------------------------------------------------
# 2d. AUGMENTED DICKEY-FULLER TEST  (paper Section 3.4 / Table 4 / Figure 12)
# ----------------------------------------------------------------------------

def adf_test_all(panel: pd.DataFrame, alpha: float = ADF_ALPHA) -> pd.DataFrame:
    """
    Replicates paper Table 4: runs the Augmented Dickey-Fuller test on each
    term's full raw series. H0: series has a unit root (non-stationary).
    Rejects H0 (i.e. series IS stationary) if p-value < alpha.
    """
    rows = []
    for term in panel.columns:
        x = panel[term].dropna().values
        try:
            adf_stat, p_value, *_ = adfuller(x, autolag="AIC")
        except Exception as e:
            adf_stat, p_value = np.nan, np.nan
        rows.append({
            "Keyword": term,
            "ADF Statistic": adf_stat,
            "p-value": p_value,
            "Stationary": "Yes" if (p_value is not np.nan and p_value < alpha) else "No",
        })
    out = pd.DataFrame(rows).sort_values("ADF Statistic").reset_index(drop=True)
    return out


def plot_adf_results(adf_df: pd.DataFrame, outpath: Path):
    """Replicates paper Figure 12: bar chart of ADF statistics, green=stationary, red=not."""
    df = adf_df.sort_values("ADF Statistic")
    colors = ["tab:green" if s == "Yes" else "tab:red" for s in df["Stationary"]]
    plt.figure(figsize=(10, max(6, 0.16 * len(df))))
    plt.bar(df["Keyword"], df["ADF Statistic"], color=colors)
    plt.xticks(rotation=90, fontsize=6)
    plt.ylabel("ADF Statistic")
    plt.title("ADF Test Results by Keyword (green = stationary at 5%)")
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()


In [19]:
# ----------------------------------------------------------------------------
# 3. SAX  (sliding-window Symbolic Aggregate approXimation)
# ----------------------------------------------------------------------------

# Faster constants reused across every keyword.
BREAKPOINTS = stats.norm.ppf(
    np.linspace(0, 1, ALPHABET_SIZE + 1)[1:-1]
)

PAA_IDX = np.linspace(
    0,
    WINDOW_SIZE,
    N_SEGMENTS + 1,
    dtype=int,
)

LETTER_TO_INT = {
    c: i
    for i, c in enumerate(ALPHABET)
}


def znormalize(x: np.ndarray) -> np.ndarray:
    mu = np.nanmean(x)
    sigma = np.nanstd(x)

    if sigma == 0 or np.isnan(sigma):
        return np.zeros_like(x)

    return (x - mu) / sigma


def paa(x: np.ndarray) -> np.ndarray:
    """
    Piecewise Aggregate Approximation using precomputed segment boundaries.
    """
    return np.array([
        x[PAA_IDX[i]:PAA_IDX[i + 1]].mean()
        for i in range(N_SEGMENTS)
    ])


def gaussian_breakpoints(alphabet_size: int) -> np.ndarray:
    quantiles = np.linspace(0, 1, alphabet_size + 1)[1:-1]
    return stats.norm.ppf(quantiles)


def value_to_symbol(value: float, breakpoints: np.ndarray = BREAKPOINTS) -> str:
    idx = np.searchsorted(breakpoints, value)
    return ALPHABET[idx]


def sax_word(window: np.ndarray) -> str:
    """
    Compute one SAX word using global breakpoints and optimized PAA.
    """
    z = znormalize(window)
    seg_means = paa(z)

    return "".join(
        value_to_symbol(v, BREAKPOINTS)
        for v in seg_means
    )


def compute_sax_words(
    series: pd.Series,
    window_size: int,
    n_segments: int,
    alphabet_size: int,
    step: int = SAX_STEP,
) -> list[str]:
    """
    Faster SAX word extraction.

    Changes vs. the original version:
    - `sliding_window_view` replaces Python start/end slicing.
    - windows are normalized in one vectorized block.
    - global breakpoints avoid repeated recomputation.
    - `step=SAX_STEP` defaults to weekly stride for daily data.
    """
    x = series.values.astype(float)

    if len(x) < window_size:
        return []

    windows = sliding_window_view(x, window_size)[::step]

    means = windows.mean(axis=1, keepdims=True)
    stds = windows.std(axis=1, keepdims=True)
    stds[stds == 0] = 1

    windows = (windows - means) / stds

    words = []

    for w in windows:
        seg_means = paa(w)
        word = "".join(
            value_to_symbol(v, BREAKPOINTS)
            for v in seg_means
        )
        words.append(word)

    return words


def sax_words_to_numeric_matrix(words: list[str]) -> np.ndarray:
    return np.array(
        [
            [LETTER_TO_INT[ch] for ch in w]
            for w in words
        ],
        dtype=np.uint8,
    )


def build_sax_feature_matrix(
    series_dict: dict[str, pd.Series],
    window_size: int,
    n_segments: int,
    alphabet_size: int,
    step: int = SAX_STEP,
) -> tuple[pd.DataFrame, None]:
    """
    Build one fixed-length SAX feature vector per keyword.

    This version skips storing every string word. It creates the sliding windows,
    normalizes them, computes numeric SAX symbols, and immediately averages the
    symbols into one per-keyword feature vector.
    """
    feature_rows = {}

    for term, s in series_dict.items():
        x = s.values.astype(float)

        if len(x) < window_size:
            continue

        windows = sliding_window_view(x, window_size)[::step]

        means = windows.mean(axis=1, keepdims=True)
        stds = windows.std(axis=1, keepdims=True)
        stds[stds == 0] = 1
        windows = (windows - means) / stds

        numeric_words = []

        for w in windows:
            seg_means = paa(w)
            numeric_words.append(
                np.searchsorted(
                    BREAKPOINTS,
                    seg_means,
                )
            )

        numeric_words = np.asarray(
            numeric_words,
            dtype=np.uint8,
        )

        feature_rows[term] = numeric_words.mean(axis=0)

    feat_df = pd.DataFrame(feature_rows).T
    feat_df.columns = [
        f"seg_{i + 1}"
        for i in range(N_SEGMENTS)
    ]

    return feat_df, None

In [20]:
# ----------------------------------------------------------------------------
# 4. eSAX (Extended SAX: mean, max, min per segment, ordered by position)
# ----------------------------------------------------------------------------

def esax_word(
    window: np.ndarray,
    n_segments: int = N_SEGMENTS,
    breakpoints: np.ndarray = BREAKPOINTS,
) -> str:
    """
    Compute one eSAX word.

    This single-window function is kept for diagnostics. The pipeline uses the
    faster vectorized `build_esax_feature_matrix` below.
    """
    z = znormalize(window)
    letters = []

    for i in range(n_segments):
        seg = z[PAA_IDX[i]:PAA_IDX[i + 1]]

        if len(seg) == 0:
            continue

        mean_v = seg.mean()
        max_v = seg.max()
        min_v = seg.min()

        positions = {
            "mean": np.argmin(np.abs(seg - mean_v)),
            "max": np.argmax(seg),
            "min": np.argmin(seg),
        }

        ordered = sorted(
            positions.items(),
            key=lambda kv: kv[1],
        )

        for key, _ in ordered:
            val = {
                "mean": mean_v,
                "max": max_v,
                "min": min_v,
            }[key]
            letters.append(value_to_symbol(val, breakpoints))

    return "".join(letters)


def compute_esax_words(
    series: pd.Series,
    window_size: int,
    n_segments: int,
    alphabet_size: int,
    step: int = SAX_STEP,
) -> list[str]:
    """
    Faster eSAX word extraction for optional diagnostics.
    """
    x = series.values.astype(float)

    if len(x) < window_size:
        return []

    windows = sliding_window_view(x, window_size)[::step]

    means = windows.mean(axis=1, keepdims=True)
    stds = windows.std(axis=1, keepdims=True)
    stds[stds == 0] = 1
    windows = (windows - means) / stds

    return [
        esax_word(window, n_segments, BREAKPOINTS)
        for window in windows
    ]


def build_esax_feature_matrix(
    series_dict: dict[str, pd.Series],
    window_size: int,
    n_segments: int,
    alphabet_size: int,
    step: int = SAX_STEP,
) -> tuple[pd.DataFrame, None]:
    """
    Build one fixed-length eSAX feature vector per keyword.

    Less computationally dense changes:
    - weekly stride by default through `SAX_STEP = 7`;
    - vectorized window construction and normalization;
    - no storage of every eSAX word string;
    - direct numeric encoding and averaging.
    """
    feature_rows = {}
    expected_len = n_segments * 3

    for term, s in series_dict.items():
        x = s.values.astype(float)

        if len(x) < window_size:
            continue

        windows = sliding_window_view(x, window_size)[::step]

        means = windows.mean(axis=1, keepdims=True)
        stds = windows.std(axis=1, keepdims=True)
        stds[stds == 0] = 1
        windows = (windows - means) / stds

        n_windows = windows.shape[0]
        numeric_words = np.empty(
            (n_windows, expected_len),
            dtype=np.uint8,
        )

        for i in range(n_segments):
            seg = windows[:, PAA_IDX[i]:PAA_IDX[i + 1]]

            mean_v = seg.mean(axis=1)
            max_v = seg.max(axis=1)
            min_v = seg.min(axis=1)

            mean_pos = np.abs(seg - mean_v[:, None]).argmin(axis=1)
            max_pos = seg.argmax(axis=1)
            min_pos = seg.argmin(axis=1)

            positions = np.column_stack([
                mean_pos,
                max_pos,
                min_pos,
            ])

            values = np.column_stack([
                mean_v,
                max_v,
                min_v,
            ])

            symbols = np.searchsorted(
                BREAKPOINTS,
                values,
            ).astype(np.uint8)

            order = np.argsort(positions, axis=1)
            row_idx = np.arange(n_windows)[:, None]

            numeric_words[:, i * 3:(i + 1) * 3] = symbols[
                row_idx,
                order,
            ]

        feature_rows[term] = numeric_words.mean(axis=0)

    feat_df = pd.DataFrame(feature_rows).T
    feat_df.columns = [
        f"pos_{i + 1}"
        for i in range(expected_len)
    ]

    return feat_df, None

In [21]:
# ----------------------------------------------------------------------------
# 5. TDA  (sliding window embedding -> Vietoris-Rips -> persistence landscapes)
# ----------------------------------------------------------------------------

def sliding_window_embedding(x: np.ndarray, dim: int, tau: int) -> np.ndarray:
    """
    Takens-style sliding window embedding.
    Returns a point cloud of shape (n_points, dim).
    """
    n = len(x)
    span = (dim - 1) * tau
    n_points = n - span
    if n_points <= 0:
        raise ValueError("Series too short for the given embedding dimension/tau.")
    cloud = np.zeros((n_points, dim))
    for i in range(dim):
        cloud[:, i] = x[i * tau: i * tau + n_points]
    return cloud


def compute_persistence_diagrams(point_cloud: np.ndarray, max_dim: int = 1):
    """Uses ripser to compute persistence diagrams up to max_dim."""
    from ripser import ripser
    result = ripser(point_cloud, maxdim=max_dim)
    return result["dgms"]  # list of arrays, one per homology dimension


def tent_function(t: np.ndarray, birth: float, death: float) -> np.ndarray:
    """Triangle/tent function for one persistence interval, evaluated at points t."""
    mid = (birth + death) / 2.0
    height = (death - birth) / 2.0
    if height <= 0:
        return np.zeros_like(t)
    val = height - np.abs(t - mid)
    return np.clip(val, 0, None)


def persistence_landscape(diagram: np.ndarray, n_points: int = N_LANDSCAPE_POINTS,
                           n_layers: int = N_LANDSCAPE_LAYERS,
                           t_range: tuple[float, float] | None = None) -> np.ndarray:
    """
    Convert a single persistence diagram (array of [birth, death] pairs) into
    a stacked persistence landscape: shape (n_layers, n_points).
    """
    diagram = diagram[np.isfinite(diagram).all(axis=1)] if len(diagram) else diagram
    if len(diagram) == 0:
        return np.zeros((n_layers, n_points))

    if t_range is None:
        t_min, t_max = diagram[:, 0].min(), diagram[:, 1].max()
    else:
        t_min, t_max = t_range
    if t_max <= t_min:
        t_max = t_min + 1.0
    t = np.linspace(t_min, t_max, n_points)

    tents = np.array([tent_function(t, b, d) for b, d in diagram])  # (n_features, n_points)
    if tents.shape[0] == 0:
        return np.zeros((n_layers, n_points))

    sorted_tents = -np.sort(-tents, axis=0)  # descending sort along feature axis
    n_available = sorted_tents.shape[0]
    landscape = np.zeros((n_layers, n_points))
    k = min(n_layers, n_available)
    landscape[:k, :] = sorted_tents[:k, :]
    return landscape


def landscape_to_vector(landscape: np.ndarray) -> np.ndarray:
    """Flatten a (n_layers, n_points) landscape into a single feature vector."""
    return landscape.flatten()


def build_tda_features(series_dict: dict[str, pd.Series], embed_dim: int, embed_tau: int,
                        max_dim: int = MAX_HOMOLOGY_DIM,
                        n_landscape_points: int = N_LANDSCAPE_POINTS,
                        n_layers: int = N_LANDSCAPE_LAYERS):
    """
    For each term:
      1. raw values -> sliding window embedding -> point cloud
      2. point cloud -> Vietoris-Rips persistent homology -> diagrams per dim
      3. diagrams -> persistence landscapes -> flattened feature vector

    Returns:
      h1_features  : DataFrame, term x flattened-H1-landscape  (loops only)
      h01_features : DataFrame, term x flattened-(H0+H1)-landscape
      diagrams_raw : dict[term] -> list of per-dimension diagrams (for plotting)
    """
    h1_rows, h01_rows, diagrams_raw = {}, {}, {}

    for term, s in series_dict.items():
        x = s.values.astype(float)
        try:
            cloud = sliding_window_embedding(x, embed_dim, embed_tau)
            dgms = compute_persistence_diagrams(cloud, max_dim=max_dim)
        except Exception as e:
            print(f"[TDA] {term} failed: {e}")
            continue

        diagrams_raw[term] = dgms

        h1_diagram = dgms[1] if len(dgms) > 1 else np.empty((0, 2))
        h1_landscape = persistence_landscape(h1_diagram, n_landscape_points, n_layers)
        h1_rows[term] = landscape_to_vector(h1_landscape)

        h0_diagram = dgms[0] if len(dgms) > 0 else np.empty((0, 2))
        combined = (
            np.vstack([h0_diagram, h1_diagram])
            if len(h0_diagram) or len(h1_diagram)
            else np.empty((0, 2))
        )
        h01_landscape = persistence_landscape(combined, n_landscape_points, n_layers)
        h01_rows[term] = landscape_to_vector(h01_landscape)

    h1_features = pd.DataFrame(h1_rows).T
    h01_features = pd.DataFrame(h01_rows).T
    return h1_features, h01_features, diagrams_raw


In [22]:
# ----------------------------------------------------------------------------
# 6. CLUSTERING  (KMeans + Hierarchical/Ward, with Silhouette & DBI)
# ----------------------------------------------------------------------------

@dataclass
class ClusterResult:
    method: str               # "KMeans" or "Hierarchical"
    representation: str        # "SAX", "eSAX", "TDA-H1", "TDA-H0H1"
    labels: pd.Series = field(default_factory=pd.Series)
    silhouette: float = np.nan
    davies_bouldin: float = np.nan


def run_kmeans(features: pd.DataFrame, n_clusters: int = N_CLUSTERS,
                random_state: int = RANDOM_STATE) -> tuple[pd.Series, float, float]:
    X = features.values
    km = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    labels = km.fit_predict(X)
    sil = silhouette_score(X, labels) if len(set(labels)) > 1 else np.nan
    dbi = davies_bouldin_score(X, labels) if len(set(labels)) > 1 else np.nan
    return pd.Series(labels, index=features.index, name="cluster"), sil, dbi


def run_hierarchical(features: pd.DataFrame, n_clusters: int = N_CLUSTERS,
                      method: str = "ward") -> tuple[pd.Series, float, float, np.ndarray]:
    X = features.values
    Z = linkage(X, method=method)
    labels = fcluster(Z, t=n_clusters, criterion="maxclust") - 1  # zero-index
    sil = silhouette_score(X, labels) if len(set(labels)) > 1 else np.nan
    dbi = davies_bouldin_score(X, labels) if len(set(labels)) > 1 else np.nan
    return pd.Series(labels, index=features.index, name="cluster"), sil, dbi, Z


def elbow_inertias(features: pd.DataFrame, k_range=range(2, 10),
                    random_state: int = RANDOM_STATE) -> pd.Series:
    inertias = {}
    for k in k_range:
        km = KMeans(n_clusters=k, random_state=random_state, n_init=10).fit(features.values)
        inertias[k] = km.inertia_
    return pd.Series(inertias)


In [23]:
# ----------------------------------------------------------------------------
# 7. PLOTTING HELPERS
# ----------------------------------------------------------------------------

def plot_elbow(inertias: pd.Series, title: str, outpath: Path):
    plt.figure(figsize=(6, 4))
    plt.plot(inertias.index, inertias.values, marker="o")
    plt.xlabel("k (number of clusters)")
    plt.ylabel("Inertia")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()


def plot_clusters_by_series(series_dict: dict[str, pd.Series], labels: pd.Series,
                             title: str, outpath: Path, n_clusters: int = N_CLUSTERS):
    fig, axes = plt.subplots(n_clusters, 1, figsize=(10, 2.2 * n_clusters), sharex=False)
    if n_clusters == 1:
        axes = [axes]
    for c in range(n_clusters):
        ax = axes[c]
        members = labels[labels == c].index
        for m in members:
            if m in series_dict:
                ax.plot(series_dict[m].index, series_dict[m].values, label=m, linewidth=0.9)
        ax.set_title(f"Cluster {c + 1} ({len(members)} terms)", fontsize=9)
        ax.legend(fontsize=6, loc="upper left", ncol=4)
    fig.suptitle(title)
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()


def plot_dendrogram(Z: np.ndarray, labels: list[str], title: str, outpath: Path):
    plt.figure(figsize=(14, 6))
    dendrogram(Z, labels=labels, leaf_rotation=90, leaf_font_size=6)
    plt.title(title)
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()


def plot_ks_bar(ks_df: pd.DataFrame, outpath: Path):
    df = ks_df.sort_values("KS Statistic")
    colors = ["tab:red" if r == "Yes" else "tab:green" for r in df["Reject H0 at 5%"]]
    plt.figure(figsize=(10, max(6, 0.18 * len(df))))
    plt.barh(df["Keyword"], df["KS Statistic"], color=colors)
    plt.xlabel("KS Statistic")
    plt.title("KS Test for Normality per Keyword (red = reject H0 at 5%)")
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()


In [24]:
# ----------------------------------------------------------------------------
# MAIN PIPELINE
# ----------------------------------------------------------------------------

def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # ---- 1. Load data ----
    print("=" * 70)
    print("STEP 1: Loading data")
    print("=" * 70)
    series_dict = load_all_series(DATA_DIR, STITCHED_SUBDIR, STITCHED_GLOB)
    panel = build_panel(series_dict)
    summary_statistics(panel).to_csv(OUTPUT_DIR / "table1_summary_statistics.csv")

    # ---- 2. KS test ----
    print("\n" + "=" * 70)
    print("STEP 2: Kolmogorov-Smirnov normality test")
    print("=" * 70)
    ks_df = ks_normality_test(panel)
    ks_df.to_csv(OUTPUT_DIR / "table3_ks_test_results.csv", index=False)
    plot_ks_bar(ks_df, OUTPUT_DIR / "fig_ks_test.png")
    print(ks_df.head(10).to_string(index=False))

    # ---- 2b. Descriptive statistics extras ----
    print("\n" + "=" * 70)
    print("STEP 2b: Descriptive statistics (boxplots, volatility, top interest)")
    print("=" * 70)
    desc_extras = descriptive_statistics_extras(panel)
    desc_extras["stds_all"].to_csv(OUTPUT_DIR / "descriptive_stds_all.csv")
    desc_extras["means_all"].to_csv(OUTPUT_DIR / "descriptive_means_all.csv")
    desc_extras["top5_volatile"].to_csv(OUTPUT_DIR / "fig10_top5_volatile.csv")
    desc_extras["top5_interest"].to_csv(OUTPUT_DIR / "fig10_top5_interest.csv")
    plot_boxplots(panel, OUTPUT_DIR / "fig9_boxplots_per_keyword.png")
    plot_top5_volatility_interest(desc_extras, OUTPUT_DIR / "fig10_top5_volatility_interest.png")
    print("Top 5 most volatile keywords:")
    print(desc_extras["top5_volatile"].to_string())
    print("Top 5 keywords by mean interest:")
    print(desc_extras["top5_interest"].to_string())
    
    # ---- 2d. ADF test ----
    print("\n" + "=" * 70)
    print("STEP 2d: Augmented Dickey-Fuller stationarity test")
    print("=" * 70)
    adf_df = adf_test_all(panel)
    adf_df.to_csv(OUTPUT_DIR / "table4_adf_test_results.csv", index=False)
    plot_adf_results(adf_df, OUTPUT_DIR / "fig12_adf_test_results.png")
    n_stationary = (adf_df["Stationary"] == "Yes").sum()
    print(f"{n_stationary}/{len(adf_df)} keywords are stationary at the 5% level.")
    print(adf_df.head(10).to_string(index=False))

    # ---- 3. SAX ----
    print("\n" + "=" * 70)
    print("STEP 3: SAX (sliding-window symbolic aggregate approximation)")
    print("=" * 70)
    sax_features, sax_words = build_sax_feature_matrix(
        series_dict, WINDOW_SIZE, N_SEGMENTS, ALPHABET_SIZE, SAX_STEP
    )
    sax_features.to_csv(OUTPUT_DIR / "sax_feature_matrix.csv")
    print(f"SAX feature matrix shape: {sax_features.shape}")

    sax_elbow = elbow_inertias(sax_features)
    plot_elbow(sax_elbow, "SAX Elbow Method", OUTPUT_DIR / "fig_sax_elbow.png")

    sax_km_labels, sax_km_sil, sax_km_dbi = run_kmeans(sax_features)
    sax_hc_labels, sax_hc_sil, sax_hc_dbi, sax_Z = run_hierarchical(sax_features)

    plot_clusters_by_series(series_dict, sax_km_labels, "SAX K-Means Clustering",
                             OUTPUT_DIR / "fig_sax_kmeans_clusters.png")
    plot_clusters_by_series(series_dict, sax_hc_labels, "SAX Hierarchical Clustering",
                             OUTPUT_DIR / "fig_sax_hierarchical_clusters.png")
    plot_dendrogram(sax_Z, list(sax_features.index), "SAX Dendrogram (Ward)",
                     OUTPUT_DIR / "fig_sax_dendrogram.png")

    print(f"SAX KMeans   -> Silhouette={sax_km_sil:.3f}, DBI={sax_km_dbi:.3f}")
    print(f"SAX Hierarch -> Silhouette={sax_hc_sil:.3f}, DBI={sax_hc_dbi:.3f}")

    # ---- 4. eSAX ----
    print("\n" + "=" * 70)
    print("STEP 4: eSAX (extended SAX: mean/max/min per segment)")
    print("=" * 70)
    esax_features, esax_words = build_esax_feature_matrix(
        series_dict, WINDOW_SIZE, N_SEGMENTS, ALPHABET_SIZE, SAX_STEP
    )
    esax_features.to_csv(OUTPUT_DIR / "esax_feature_matrix.csv")
    print(f"eSAX feature matrix shape: {esax_features.shape}")

    esax_elbow = elbow_inertias(esax_features)
    plot_elbow(esax_elbow, "eSAX Elbow Method", OUTPUT_DIR / "fig_esax_elbow.png")

    esax_km_labels, esax_km_sil, esax_km_dbi = run_kmeans(esax_features)
    esax_hc_labels, esax_hc_sil, esax_hc_dbi, esax_Z = run_hierarchical(esax_features)

    plot_clusters_by_series(series_dict, esax_km_labels, "eSAX K-Means Clustering",
                             OUTPUT_DIR / "fig_esax_kmeans_clusters.png")
    plot_clusters_by_series(series_dict, esax_hc_labels, "eSAX Hierarchical Clustering",
                             OUTPUT_DIR / "fig_esax_hierarchical_clusters.png")
    plot_dendrogram(esax_Z, list(esax_features.index), "eSAX Dendrogram (Ward)",
                     OUTPUT_DIR / "fig_esax_dendrogram.png")

    print(f"eSAX KMeans   -> Silhouette={esax_km_sil:.3f}, DBI={esax_km_dbi:.3f}")
    print(f"eSAX Hierarch -> Silhouette={esax_hc_sil:.3f}, DBI={esax_hc_dbi:.3f}")

    # ---- 5. TDA ----
    print("\n" + "=" * 70)
    print("STEP 5: TDA (Vietoris-Rips persistent homology + landscapes)")
    print("=" * 70)
    tda_h1, tda_h01, tda_diagrams = build_tda_features(
        series_dict, EMBED_DIM, EMBED_TAU, MAX_HOMOLOGY_DIM,
        N_LANDSCAPE_POINTS, N_LANDSCAPE_LAYERS
    )
    tda_h1.to_csv(OUTPUT_DIR / "tda_h1_feature_matrix.csv")
    tda_h01.to_csv(OUTPUT_DIR / "tda_h0h1_feature_matrix.csv")
    print(f"TDA H1 feature matrix shape: {tda_h1.shape}")
    print(f"TDA H0+H1 feature matrix shape: {tda_h01.shape}")

    # 5a. KMeans on H1-only landscapes (paper's first TDA strategy)
    tda_km_labels, tda_km_sil, tda_km_dbi = run_kmeans(tda_h1)
    plot_clusters_by_series(series_dict, tda_km_labels, "TDA K-Means Clustering (H1 loops)",
                             OUTPUT_DIR / "fig_tda_kmeans_clusters.png")
    print(f"TDA KMeans (H1)        -> Silhouette={tda_km_sil:.3f}, DBI={tda_km_dbi:.3f}")

    # 5b. Hierarchical on combined H0+H1 landscapes (paper's second TDA strategy)
    tda_hc_labels, tda_hc_sil, tda_hc_dbi, tda_Z = run_hierarchical(tda_h01)
    plot_clusters_by_series(series_dict, tda_hc_labels,
                             "TDA Hierarchical Clustering (H0+H1)",
                             OUTPUT_DIR / "fig_tda_hierarchical_clusters.png")
    plot_dendrogram(tda_Z, list(tda_h01.index), "TDA Dendrogram (Ward, H0+H1)",
                     OUTPUT_DIR / "fig_tda_dendrogram.png")
    print(f"TDA Hierarchical (H0+H1) -> Silhouette={tda_hc_sil:.3f}, DBI={tda_hc_dbi:.3f}")

    # ---- 6. Summary comparison table (paper Table 6) ----
    print("\n" + "=" * 70)
    print("STEP 6: Final comparison table")
    print("=" * 70)
    comparison = pd.DataFrame({
        "SAX": {
            "Silhouette (KMeans)": sax_km_sil, "DBI (KMeans)": sax_km_dbi,
            "Silhouette (Hierarchical)": sax_hc_sil, "DBI (Hierarchical)": sax_hc_dbi,
        },
        "eSAX": {
            "Silhouette (KMeans)": esax_km_sil, "DBI (KMeans)": esax_km_dbi,
            "Silhouette (Hierarchical)": esax_hc_sil, "DBI (Hierarchical)": esax_hc_dbi,
        },
        "TDA": {
            "Silhouette (KMeans)": tda_km_sil, "DBI (KMeans)": tda_km_dbi,
            "Silhouette (Hierarchical)": tda_hc_sil, "DBI (Hierarchical)": tda_hc_dbi,
        },
    })
    comparison.to_csv(OUTPUT_DIR / "table6_clustering_comparison.csv")
    print(comparison.to_string())

    # ---- 7. Save cluster assignments ----
    all_labels = pd.DataFrame({
        "SAX_KMeans": sax_km_labels,
        "SAX_Hierarchical": sax_hc_labels,
        "eSAX_KMeans": esax_km_labels,
        "eSAX_Hierarchical": esax_hc_labels,
        "TDA_KMeans_H1": tda_km_labels,
        "TDA_Hierarchical_H0H1": tda_hc_labels,
    })
    all_labels.to_csv(OUTPUT_DIR / "all_cluster_assignments.csv")

    print(f"\nAll outputs written to: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()


STEP 1: Loading data
[load_all_series] Loaded 191 term series.

STEP 2: Kolmogorov-Smirnov normality test
                                 Keyword  KS Statistic       p-value Reject H0 at 5%
                     men‘s_march_madness      0.523194  0.000000e+00             Yes
                      groundhog_day_2023      0.513171  0.000000e+00             Yes
2024_united_states_presidential_election      0.509291  0.000000e+00             Yes
                arizona_election_results      0.492111  0.000000e+00             Yes
                     full_moon_june_2022      0.491860 1.903010e-313             Yes
                    australian_open_2022      0.478355  0.000000e+00             Yes
             when_we_were_young_festival      0.451864 4.925822e-301             Yes
                             fathers_day      0.444376 1.245987e-290             Yes
                         canelo_vs_bivol      0.443480 2.116628e-289             Yes
                        st_patrick's_day    